In [3]:
import pandas as pd
import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k

/opt/anaconda3/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [4]:
# Cell 1: imports and corrected helper
import pandas as pd
from pathlib import Path

def build_interactions(behaviors_path: Path) -> pd.DataFrame:
    """
    Parse a MIND behaviors.tsv file into a DataFrame of (user_id, article_id)
    pairs for every click—both from the user’s click history and from
    impressions where label==1.
    The MIND behaviors.tsv format is:
      impression_id \t user_id \t timestamp \t history \t impressions
    """
    records = []
    with behaviors_path.open('r', encoding='utf-8') as f:
        for line in f:
            # split into exactly 5 parts
            parts = line.strip().split('\t')
            if len(parts) != 5:
                # skip malformed lines
                continue

            _, user_id, _, history, impressions = parts

            # 1) clicks in the “history” field
            if history != '-':
                for nid in history.split():
                    records.append((user_id, nid))

            # 2) clicks in the “impressions” field (nid-label)
            for imp in impressions.split():
                nid, label = imp.rsplit('-', 1)
                if label == '1':
                    records.append((user_id, nid))

    return pd.DataFrame(records, columns=['user_id', 'article_id'])


In [6]:
base      = Path('/Users/harshadayiniakula/Desktop/RS')
train_beh = base / 'MINDsmall_train' / 'behaviors.tsv'
dev_beh   = base / 'MINDsmall_dev'   / 'behaviors.tsv'

train_df = build_interactions(train_beh)
dev_df   = build_interactions(dev_beh)

# Sanity checks
print("Train file exists:", train_beh.exists())
print("Dev  file exists:", dev_beh.exists())
print("Train interactions:", train_df.shape)
print("Dev interactions:  ", dev_df.shape)

Train file exists: True
Dev  file exists: True
Train interactions: (5343983, 2)
Dev interactions:   (2473897, 2)


In [8]:
# Cell 3: save to CSV for LightFM ingestion
train_df.to_csv('train_interactions.csv', index=False)
dev_df.to_csv('val_interactions.csv',   index=False)

print(f"Saved {len(train_df)} train interactions → train_interactions.csv")
print(f"Saved {len(dev_df)}   val interactions → val_interactions.csv")


Saved 5343983 train interactions → train_interactions.csv
Saved 2473897   val interactions → val_interactions.csv


In [5]:
train_df = pd.read_csv("train_interactions.csv")  # must have user_id, article_id
dev_df   = pd.read_csv("val_interactions.csv")

In [7]:
val_df_filtered = dev_df.copy()  

# Build the user/item vocab lists
all_users = pd.concat([train_df['user_id'], val_df_filtered['user_id']]).unique().tolist()
all_items = pd.concat([train_df['article_id'], val_df_filtered['article_id']]).unique().tolist()

# Making new column with relevant Verticals and SubVerticals

In [9]:
news = pd.read_csv(
    'MINDsmall_train/news.tsv',
    sep='\t', header=None,
    names=[
        'newid','vertical','subvertical',
        'title','abstract','url',
        'title_entities','abstract_entities'
    ],
    dtype=str
)

In [11]:
verticals = news['vertical'].unique().tolist()


In [13]:
print(verticals)

['lifestyle', 'health', 'news', 'sports', 'weather', 'entertainment', 'autos', 'travel', 'foodanddrink', 'tv', 'finance', 'movies', 'video', 'music', 'kids', 'middleeast', 'northamerica']


In [15]:
# mind_df was your English news DataFrame with a "subvertical" column
sub_counts = news["subvertical"].value_counts()
# Show the top 30 most common subverticals
top30 = sub_counts.head(30)
print(top30)
# Optionally export for manual inspection:
top30.to_csv("top30_subverticals.csv")


subvertical
newsus                      6564
football_nfl                5420
newspolitics                2826
newscrime                   2254
weathertopstories           2047
newsworld                   1720
football_ncaa               1665
baseball_mlb                1661
basketball_nba              1555
newsscienceandtechnology    1210
news                        1185
newstrends                  1176
more_sports                 1065
travelarticle               1042
travelnews                   902
lifestylebuzz                894
autosnews                    837
basketball_ncaa              774
financenews                  697
finance-real-estate          584
finance-companies            567
icehockey_nhl                531
medical                      479
recipes                      463
health-news                  459
golf                         446
mma                          437
musicnews                    414
markets                      410
newsoffbeat                  40

In [17]:
# 1.2 Find every unique subvertical that starts with "news"
all_subs = news["subvertical"].unique().tolist()
news_subs = sorted([sv for sv in all_subs if sv.startswith("news")])
print(news_subs)


['news', 'newsbusiness', 'newscrime', 'newselection2020', 'newsfactcheck', 'newsgoodnews', 'newsnational', 'newsoffbeat', 'newsopinion', 'newsother', 'newsphotos', 'newspolitics', 'newsrealestate', 'newsscience', 'newsscienceandtechnology', 'newstrends', 'newstvmedia', 'newsus', 'newsvideo', 'newsweather', 'newsworld', 'newsworldpolitics']


In [83]:
print(news['subvertical'].nunique())

264


In [19]:

news_subs = [
    'news', 'newsbusiness', 'newscrime', 'newselection2020', 'newsfactcheck',
    'newsgoodnews', 'newsnational', 'newsoffbeat', 'newsopinion', 'newsother',
    'newsphotos', 'newspolitics', 'newsrealestate', 'newsscience',
    'newsscienceandtechnology', 'newstrends', 'newstvmedia', 'newsus',
    'newsvideo', 'newsweather', 'newsworld', 'newsworldpolitics'
]

# 1) Compute frequencies
sub_counts = news['subvertical'].value_counts()

# 2) Build a dict and sort descending
freqs = {cat: sub_counts.get(cat, 0) for cat in news_subs}
sorted_freqs = sorted(freqs.items(), key=lambda x: x[1], reverse=True)

# 3) Print in descending order
for cat, freq in sorted_freqs:
    print(f"{cat}: {freq}")

newsus: 6564
newspolitics: 2826
newscrime: 2254
newsworld: 1720
newsscienceandtechnology: 1210
news: 1185
newstrends: 1176
newsoffbeat: 405
newsopinion: 315
newsgoodnews: 176
newsbusiness: 48
newsphotos: 22
newsfactcheck: 17
newsscience: 15
newsweather: 11
newselection2020: 8
newsworldpolitics: 5
newsnational: 1
newsother: 1
newsrealestate: 1
newstvmedia: 1
newsvideo: 1


In [21]:
# 1) Identify all “news…” subverticals
news_children = [sv for sv in news["subvertical"].unique() if sv.startswith("news")]

# 2) Build new_category:
#    – if the article’s subvertical is one of the news children, use that
#    – otherwise fall back to the original vertical
news["new_category"] = news["subvertical"].where(
    news["subvertical"].isin(news_children),
    news["vertical"]
)

# 3) (Optional) Inspect your new set of categories and their counts
cats = news["new_category"].value_counts().sort_values(ascending=False)
print(cats)


new_category
sports                      14510
newsus                       6564
finance                      3107
newspolitics                 2826
lifestyle                    2479
travel                       2349
newscrime                    2254
weather                      2048
health                       1885
newsworld                    1720
autos                        1639
foodanddrink                 1376
news                         1358
newsscienceandtechnology     1210
newstrends                   1176
video                         900
tv                            889
music                         769
movies                        606
entertainment                 570
newsoffbeat                   405
newsopinion                   315
newsgoodnews                  176
newsbusiness                   48
newsphotos                     22
newsfactcheck                  17
kids                           17
newsscience                    15
newsweather                    11
n

In [25]:
# Cell: Sample & print a few “newsus” articles for manual inspection
import random

# Filter MIND to only newsus subvertical
newsus_df = news[news["subvertical"] == "news"]

# Sample 5 random rows (or fewer, if there aren’t that many)
sample_n = min(5, len(newsus_df))
for row in newsus_df.sample(sample_n, random_state=42).itertuples():
    print("ID       :", row.newid)
    print("Title    :", row.title)
    print("Abstract :", row.abstract, "\n" + "-"*60 + "\n")


ID       : N9990
Title    : Tapper goes after GOP contradictions on impeachment inquiry
Abstract : House Democrats announced the formal impeachment inquiry into President Donald Trump one month ago, and it has been an eventful month. CNN's Jake Tapper recaps, and highlights recent GOP protests to the inquiry. 
------------------------------------------------------------

ID       : N26506
Title    : Rep. Karen Bass on the first public impeachment inquiry hearing
Abstract : California Democrat Rep. Karen Bass says it is very important for the American people to hear directly from the witnesses. 
------------------------------------------------------------

ID       : N45120
Title    : New worries for American held in Lebanon
Abstract : The plea to Pres. Trump from Amer Fakhoury's family. 
------------------------------------------------------------

ID       : N2293
Title    : Ex-inmate Cyntoia Brown-Long argues for redemption in memoir
Abstract : Cyntoia Brown-Long's case drew celebrit

In [27]:
# Cell: Sample & print a few “newsus” articles for manual inspection
import random

# Filter MIND to only newsus subvertical
newsus_df = news[news["subvertical"] == "newsus"]

# Sample 5 random rows (or fewer, if there aren’t that many)
sample_n = min(5, len(newsus_df))
for row in newsus_df.sample(sample_n, random_state=42).itertuples():
    print("ID       :", row.newid)
    print("Title    :", row.title)
    print("Abstract :", row.abstract, "\n" + "-"*60 + "\n")


ID       : N60943
Title    : Car trailer hauling 5 vehicles, including Porsche, stolen in the Bronx
Abstract : nan 
------------------------------------------------------------

ID       : N46417
Title    : Elk Fire burning more than 600 acres, 40% contained
Abstract : The Elk Fire is burning near Boy Scout Ranch, Glacier View in Colorado 
------------------------------------------------------------

ID       : N20626
Title    : Schools pushed into delay/closing decisions as winter arrives early
Abstract : Winter weather has arrived. How do school superintendents decide to stay on regular schedule, operate on a delay or close? 
------------------------------------------------------------

ID       : N981
Title    : CBP agents wrote fake court dates on paperwork to send migrants back to Mexico, records show
Abstract : SAN DIEGO - Asylum-seekers who have finished their court cases are being sent back to Mexico with documents that contain fraudulent future court dates, keeping some migran

news category in subvertical makes little sense to me, it has politics related news that couldve been in newspolitics itself a
newsus is news containing usa states. I think it is best I club it into newsworld to make it compatible for my telugu news recommender system 

In [30]:
# 2) Build a remapping dict that only overrides 'newsus'
sub2cat = {sv: sv for sv in news_children}  # start as identity
sub2cat['newsus'] = 'newsworld'             # only change 'newsus'

# 3) Create new_category:
#    if its subvertical is in news_children, use sub2cat[subvertical]
#    otherwise fall back to the original vertical
news['new_category'] = news['subvertical'].map(sub2cat) \
                                .fillna(news['vertical'])

# 4) Inspect your new frequencies
print(news['new_category'].value_counts().sort_values(ascending=False))


new_category
sports                      14510
newsworld                    8284
finance                      3107
newspolitics                 2826
lifestyle                    2479
travel                       2349
newscrime                    2254
weather                      2048
health                       1885
autos                        1639
foodanddrink                 1376
news                         1358
newsscienceandtechnology     1210
newstrends                   1176
video                         900
tv                            889
music                         769
movies                        606
entertainment                 570
newsoffbeat                   405
newsopinion                   315
newsgoodnews                  176
newsbusiness                   48
newsphotos                     22
newsfactcheck                  17
kids                           17
newsscience                    15
newsweather                    11
newselection2020                8
n

In [66]:
new_cats = set(news["new_category"].unique())
missing_feats = new_cats - set(verticals)
print("Subcategories never registered as features:", missing_feats)

Subcategories never registered as features: {'newsworld', 'newstrends', 'newscrime', 'newspolitics', 'newsscienceandtechnology'}


In [32]:
# Cell: Sample & print a few “newsus” articles for manual inspection
import random

# Filter MIND to only newsus subvertical
newsus_df = news[news["vertical"] == "midleeast"]

# Sample 5 random rows (or fewer, if there aren’t that many)
sample_n = min(5, len(newsus_df))
for row in newsus_df.sample(sample_n, random_state=42).itertuples():
    print("ID       :", row.newid)
    print("Title    :", row.title)
    print("Abstract :", row.abstract, "\n" + "-"*60 + "\n")


In [85]:
# Cell: Sample & print a few “newsus” articles for manual inspection
import random

# Filter MIND to only newsus subvertical
newsus_df = news[news["vertical"] == "northamerica"]

# Sample 5 random rows (or fewer, if there aren’t that many)
sample_n = min(5, len(newsus_df))
for row in newsus_df.sample(sample_n, random_state=42).itertuples():
    print("ID       :", row.newid)
    print("Title    :", row.title)
    print("Abstract :", row.abstract, "\n" + "-"*60 + "\n")


ID       : N57229
Title    : Republicans have floated 17 different defenses of Trump's Ukraine actions
Abstract : Since the House formally launched the impeachment inquiry of President Trump, congressional Republicans have floated no fewer than 17 different defenses of his actions on Ukraine. 
------------------------------------------------------------



In [89]:
# Cell: Sample & print a few “newsus” articles for manual inspection
import random

# Filter MIND to only newsus subvertical
newsus_df = news[news["subvertical"] == "newsscience"]

# Sample 5 random rows (or fewer, if there aren’t that many)
sample_n = min(5, len(newsus_df))
for row in newsus_df.sample(sample_n, random_state=42).itertuples():
    print("ID       :", row.newid)
    print("Title    :", row.title)
    print("Abstract :", row.abstract, "\n" + "-"*60 + "\n")


ID       : N39349
Title    : Interstellar space even weirder than expected, NASA probe reveals
Abstract : The spacecraft is just the second ever to venture beyond the boundary that separates us from the rest of the galaxy. 
------------------------------------------------------------

ID       : N34480
Title    : The Leonid Meteor Shower Will Light up the Sky Monday Night
Abstract : Sky-gazers in both the northern and southern hemispheres can look up together for this celestial spectacle. 
------------------------------------------------------------

ID       : N49402
Title    : 11,000 scientists sign declaration of global climate emergency
Abstract : Thousands of scientists around the world have mobilized to warn people of an impending global climate emergency if levels of greenhouse gas emissions are not reduced. 
------------------------------------------------------------

ID       : N49948
Title    : How the world's most widely used insecticide led to a fishery collapse
Abstract :

In [97]:
# Cell: Sample & print a few “newsus” articles for manual inspection
import random

# Filter MIND to only newsus subvertical
newsus_df = news[news["subvertical"] == "newsrealestate"]

# Sample 5 random rows (or fewer, if there aren’t that many)
sample_n = min(5, len(newsus_df))
for row in newsus_df.sample(sample_n, random_state=42).itertuples():
    print("ID       :", row.newid)
    print("Title    :", row.title)
    print("Abstract :", row.abstract, "\n" + "-"*60 + "\n")


ID       : N24131
Title    : Berlin Freezes Rents in Landmark Plan to Tackle Cost Spiral
Abstract : Berlin's governing parties struck a deal to freeze rents for five years, marking one of the most radical plans to tackle spiraling housing costs in a major city and hitting the shares of major apartment owners. 
------------------------------------------------------------



In [107]:
import random
newsus_df = news[news["subvertical"] == "video"]
sample_n = min(5, len(newsus_df))
for row in newsus_df.sample(sample_n, random_state=42).itertuples():
    print("ID       :", row.newid)
    print("Title    :", row.title)
    print("Abstract :", row.abstract, "\n" + "-"*60 + "\n")


ID       : N10827
Title    : Will Smith, Helen Mirren and Chris Martin Join the World's Big Sleep Out
Abstract : On Saturday, Dec. 7, the world will unite to help fight homelessness. Get the details. 
------------------------------------------------------------

ID       : N44270
Title    : Kylie Jenner and Travis Scott Reportedly Taking a Break After More Than 2 Years of Dating
Abstract : Kylie Jenner and Travis Scott, who were first linked in April 2017, are reportedly taking a break. 
------------------------------------------------------------

ID       : N34986
Title    : We the People: The White House
Abstract : The president of the United States serves as the chief executive and commander of the armed forces, all defined in Article II of the Constitution as the executive branch. 
------------------------------------------------------------

ID       : N20766
Title    : 4 Things to Know About Keanu Reeves' New Girlfriend
Abstract : Keanu is single no more! From philanthropist to 

In [35]:
# 1) Define your rare→common mapping
rare_to_common = {
    
    "middleeast": "newsworld", #checked manually
    "northamerica": "newspolitics", #trump related news about impeachment in relation to ukraine's actions (i think it fits newspolitics more than newsworld) 
    "newsnational": "newsworld", #news about lifestyle but the abstract is not clear enough, i ll let it be in newsworld
    "newsus" : "newsworld",
   
    "newselection2020": "newspolitics",
    "newsworldpolitics": "newspolitics",
   
    "newsscience": "newsscienceandtechnology",
    
    "newsrealestate": "newsworld", #Berlin Freezes Rents in Landmark Plan to Tackle Cost Spiral
   
    "newsweather": "weather",
    
    "newsfactcheck": "news",
    "newsother":      "news",
    
    "kids": "lifestyle",
   
    "newsvideo":   "video",
    "newstvmedia": "video",

    "newsoffbeat" : "news",                   
    "newsopinion" : "news",                
    "newsgoodnews"  : "news",               
    "newsbusiness"  : "news" ,               
    "newsphotos"    : "news"  ,              
}

news["new_category"] = news["new_category"].replace(rare_to_common)

 (Optional) Re‐inspect the updated frequencies
print(news["new_category"].value_counts().sort_values(ascending=False))


new_category
sports                      14510
newsworld                    8288
finance                      3107
newspolitics                 2840
lifestyle                    2496
travel                       2349
news                         2342
newscrime                    2254
weather                      2059
health                       1885
autos                        1639
foodanddrink                 1376
newsscienceandtechnology     1225
newstrends                   1176
video                         902
tv                            889
music                         769
movies                        606
entertainment                 570
Name: count, dtype: int64


In [55]:
null_count = news["new_category"].isna().sum()
print(f"Articles in `news` with no new_category: {null_count}")

# 👉 2) Which interaction-items aren’t even in `news`?
fit_items   = set(ds.mapping()[1].keys())      # the IDs you passed to ds.fit(...)
news_items  = set(news["newid"])               # the IDs in your news table
missing_ids = fit_items - news_items
print(f"Items in interactions but missing from news table: {len(missing_ids)}")
print(sorted(missing_ids)[:20], "...")         # show a few examples


Articles in `news` with no new_category: 0
Items in interactions but missing from news table: 94057
['U1', 'U10', 'U100', 'U1000', 'U10000', 'U10001', 'U10002', 'U10003', 'U10004', 'U10005', 'U10006', 'U10007', 'U10008', 'U10009', 'U1001', 'U10010', 'U10011', 'U10012', 'U10013', 'U10014'] ...


In [67]:
news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   newid              51282 non-null  object
 1   vertical           51282 non-null  object
 2   subvertical        51282 non-null  object
 3   title              51282 non-null  object
 4   abstract           48616 non-null  object
 5   url                51282 non-null  object
 6   title_entities     51279 non-null  object
 7   abstract_entities  51278 non-null  object
 8   new_category       51282 non-null  object
dtypes: object(9)
memory usage: 3.5+ MB


In [69]:
news.to_csv("newcat.csv", index=False)


# Feature Building

In [27]:
from lightfm.data import Dataset as LFDataset


In [41]:
new_cats = news["new_category"].unique().tolist()
#verticals = news['vertical'].unique().tolist()


In [43]:
print(new_cats)

['lifestyle', 'health', 'newsworld', 'sports', 'weather', 'entertainment', 'newsscienceandtechnology', 'autos', 'travel', 'newspolitics', 'news', 'foodanddrink', 'tv', 'finance', 'movies', 'newstrends', 'video', 'music', 'newscrime']


In [45]:
ds = Dataset()
ds.fit(
    users=all_users,
    items=all_items,
    user_features=[],
    item_features=new_cats
)

In [47]:
train_interactions, _ = ds.build_interactions(
    (row.user_id, row.article_id) for row in train_df.itertuples()
)
val_interactions, _   = ds.build_interactions(
    (row.user_id, row.article_id) for row in val_df_filtered.itertuples()
)

In [53]:
newcat_map = news.set_index('newid')['new_category'].to_dict()

# 2) Create (item_id, [feature_name]) tuples for items in your split
item_feature_tuples = [
    (item, [newcat_map[item]])      # <-- wrap the vertical in a list!
    for item in all_items
    if item in newcat_map
]

# 3) Build the sparse CSR matrix of shape (n_items × 17)
item_features = ds.build_item_features(item_feature_tuples)

In [81]:
from scipy import sparse

# After building item_features_new using LFDataset.build_item_features(...)
sparse.save_npz("item_features_newcat.npz", item_features)

# Model Training

In [45]:
model = LightFM(
    no_components=30,
    loss='warp',        # try 'bpr' or 'logistic' also in HPO
    user_alpha=1e-6,
    item_alpha=1e-6
)

30 latent embedding vectors in the model : this is because we already have 17 explicit features, giving a bit more space will allow it to learn hidden factors or dimensions better.

In [47]:
# Cell 7: Train the model, passing in your new item_features
model.fit(
    train_interactions,
    item_features=item_features,
    epochs=10,
    num_threads=4,
    verbose=True
)

Epoch: 100%|████████████████████████████████████| 10/10 [01:05<00:00,  6.54s/it]


In [48]:
# Cell 8: Evaluate on validation with Precision@10
val_precision = precision_at_k(
    model,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

print(f'Validation Precision@10: {val_precision:.4f}')

Validation Precision@10: 0.0686


In [51]:
# Cell 8: Compute & print a suite of metrics on both train and val
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# 1) Precision@10
train_prec = precision_at_k(
    model,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

In [52]:
# Cell 8: Compute & print a suite of metrics on both train and val
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# 2) Recall@10
train_rec = recall_at_k(
    model,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

val_rec = recall_at_k(
    model,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

# 3) AUC Score
train_auc = auc_score(
    model,
    train_interactions,
    item_features=item_features
).mean()

val_auc = auc_score(
    model,
    val_interactions,
    item_features=item_features
).mean()

print(f"Train Precision@10: {train_prec:.4f}    |  Val Precision@10: {val_prec:.4f}")
print(f"Train Recall@10:    {train_rec:.4f}    |  Val Recall@10:    {val_rec:.4f}")
print(f"Train AUC:          {train_auc:.4f}    |  Val AUC:          {val_auc:.4f}")


NameError: name 'val_prec' is not defined

In [57]:
print(f"Train Precision@10: {train_prec:.4f}    |  Val Precision@10: {val_precision:.4f}")
print(f"Train Recall@10:    {train_rec:.4f}    |  Val Recall@10:    {val_rec:.4f}")
print(f"Train AUC:          {train_auc:.4f}    |  Val AUC:          {val_auc:.4f}")


Train Precision@10: 0.0805    |  Val Precision@10: 0.0686
Train Recall@10:    0.0288    |  Val Recall@10:    0.0269
Train AUC:          0.9773    |  Val AUC:          0.8706


# NO REGULARISATION BASIC MODEL

In [59]:
model1 = LightFM(
    no_components = 30,
    loss = 'warp',
)

In [61]:
model1.fit(
    train_interactions,
    item_features=item_features,
    epochs=20,
    num_threads=4,
    verbose=True
)

Epoch: 100%|████████████████████████████████████| 20/20 [03:19<00:00,  9.95s/it]


In [62]:
# Cell 8: Compute & print a suite of metrics on both train and val
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

# 1) Precision@10
train_prec = precision_at_k(
    model1,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

val_prec = precision_at_k(
    model1,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

# 2) Recall@10
train_rec = recall_at_k(
    model1,
    train_interactions,
    item_features=item_features,
    k=10
).mean()

val_rec = recall_at_k(
    model1,
    val_interactions,
    item_features=item_features,
    k=10
).mean()

# 3) AUC Score
train_auc = auc_score(
    model1,
    train_interactions,
    item_features=item_features
).mean()

val_auc = auc_score(
    model1,
    val_interactions,
    item_features=item_features
).mean()

print(f"Train Precision@10: {train_prec:.4f}    |  Val Precision@10: {val_prec:.4f}")
print(f"Train Recall@10:    {train_rec:.4f}    |  Val Recall@10:    {val_rec:.4f}")
print(f"Train AUC:          {train_auc:.4f}    |  Val AUC:          {val_auc:.4f}")


Train Precision@10: 0.0872    |  Val Precision@10: 0.0735
Train Recall@10:    0.0320    |  Val Recall@10:    0.0282
Train AUC:          0.9844    |  Val AUC:          0.8727


In [63]:
# Cell: Save your trained LightFM model to disk

import pickle

model_path = "lightfm_model_newcat.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model1, f)

print(f"Saved LightFM model to {model_path}")


Saved LightFM model to lightfm_model_newcat.pkl


In [71]:
print(news['new_category'].nunique())

19


In [77]:
new_cats = sorted(news["new_category"].unique().tolist())


In [79]:
with open("new_cats.txt", "w") as f:
    f.writelines([cat + "\n" for cat in new_cats])